# NDSH HFF experiment

- Dataset: `Datasets/GEFCOM2017/GEF2017_univarivate.csv`
- Hierarchy mode: `dedup`
- Parameters and evaluation logic are copied from the corresponding read-only `/Users/yingjiezhao/yjz/yjzPy/FHF` notebook.
- All generated artifacts are written into this experiment folder.


# HFF fh3

Hierarchy-aware federated forecasting configuration for HFF.


In [ ]:
from __future__ import annotations
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import sys
import time
from pathlib import Path

project_dir = Path.cwd()
module_dir = PROJECT_ROOT / "PPHF_modules" / "fhf"

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from PPHF_modules.fhf.FHF_HF2 import HFLArgs, run_hfl_with_third_party_reconciliation


In [ ]:

print('torch version:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('working directory:', project_dir)


In [ ]:

# -------------------------
# Parameter setup
# -------------------------
args = HFLArgs()

# Uniform sampling without replacement. 1.0 preserves the original run.
args.sampling_rate = 1.0
args.sampling_seed = None  # Defaults to args.seed

# Change this path to your own dataset if needed.
# Example (current folder):
args.csv_path = str((PROJECT_ROOT / 'Datasets/GEFCOM2017/GEF2017_univarivate.csv').resolve())
args.reject_duplicate_series = True

# Basic experiment settings
args.model_type = 'lstm'
args.truncated = None # '2013-05-01'
args.fh = 2
args.lags = 48
args.test_ratio = 0.2
args.batch_size = 256

# FL / HFL settings
args.num_rounds = 100
args.edge_agg_rounds = 1
args.edge_agg_rounds_by_edge = None
args.edge_execution_mode = 'sequential'   # each edge finishes all its own rounds first
args.local_epochs = 1
args.client_frac_per_edge = 1.0


# Early stopping / stop criteria
args.early_stop_enabled = True
'''
args.early_stop_tol = 1e-4
args.early_stop_metric = 'avg_normalised_loss_delta'
args.min_rounds = 100'''
args.early_stop_tol = 1e-5
args.early_stop_metric = "avg_normalised_loss_delta"
args.min_rounds = 100
args.edge_trainable = False
args.cloud_trainable = False
args.hier_train_mode = "self_train_then_aggregate"

# Event-level DP O2MR artifact settings.
# The main run remains non-private; reusable P matrices and train-derived
# one shared training-target time-slot bounds file is exported after evaluation.
args.dp_enabled = False
args.dp_aware_reconciliation = False
args.dp_privacy_level = "event"
args.dp_mechanism = "laplace"
args.dp_epsilon = 1.0
args.dp_delta = 1e-5
args.dp_bounds_source = "common_local_training_targets"
args.dp_bounds_scope = "per_client_time_slot"
args.dp_slots_per_day = 48
args.dp_lower_quantile = 0.01
args.dp_upper_quantile = 0.99
args.dp_iqr_margin = 0.0
args.dp_physical_lower = 0.0

# Reconciliation settings
args.recon_method = ('bu', 'mint_cov', 'mint_shrinkage', 'mint_ols', 'mint_var', 'wls_structure')
args.validate_p2p = False
args.p2p_atol = 1e-7

# Runtime / logging
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
args.verbose_cloud_rounds = True
args.verbose_edge_rounds = False
args.print_every = 10
args.seed = 42
args.torch_num_threads = 1

args


In [ ]:

# Run HFF
result = run_hfl_with_third_party_reconciliation(args)

nodes = result['nodes']
layout = result['layout']
cid2series = result['cid2series']
cloud_round_logs = result['cloud_round_logs']
edge_round_logs = result['edge_round_logs']
timings = result['timings']

print('Run finished.')
print('Number of hierarchy nodes:', len(layout.node_ids))
print('Cloud node id:', layout.cloud_id)
print('Edge ids:', layout.edge_ids)
print('Number of client nodes:', len(layout.client_ids))


In [ ]:
# Timing breakdown: setup, FL training, reconciliation, and evaluation
# evaluation_sec is updated after the evaluation cell below.
def make_timing_df(timings):
    ordered_modules = [
        'setup_sec',
        'data_preparation_sec',
        'model_server_initialization_sec',
        'fl_training_sec',
        'residual_collection_sec',
        'recon_matrix_sec',
        'p2p_reconcile_sec',
        'reconciliation_sec',
        'evaluation_sec',
        'total_sec',
    ]
    return pd.DataFrame(
        [{'module': key, 'seconds': timings.get(key, np.nan)} for key in ordered_modules]
    )

# At this point evaluation has not been run, so evaluation_sec is still 0.0.
timing_df = make_timing_df(timings)
timing_df


In [ ]:
# Round logs, including average training loss and early-stopping status
cloud_logs_df = pd.DataFrame(cloud_round_logs)
edge_logs_df = pd.DataFrame(edge_round_logs)

print('Cloud round logs:')
display(cloud_logs_df.tail())
print('Edge round logs:')
display(edge_logs_df.head())

print('Early stop summary:')
display(pd.DataFrame([result.get('early_stop', {})]))


In [ ]:
# Last few cloud rounds show convergence and whether early stopping was triggered.
cloud_logs_df.tail()


In [ ]:

# Quick inspection of saved forecasts
node0 = layout.node_ids[0]
print('example node id:', node0)
print('series name:', cid2series[node0])
print('methods:', list(nodes[node0].reconciled_forecasts.keys()))
for method, arr in nodes[node0].reconciled_forecasts.items():
    print(method, np.asarray(arr).shape)



## Rebuild forecast tables for evaluation

This section mirrors the idea in `FHF_fh1.ipynb`, but uses the outputs already stored in the HFF nodes.


In [ ]:

forecast_horizon = ['y'] + [f'post_{i}' for i in range(1, args.fh + 1)]
node_order = list(layout.node_ids)


def split_node_df(df_one, args):
    sub = df_one.sort_values('ds').copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split for node: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    tr = sub.iloc[:n_train_windows].copy()
    ts = sub.iloc[n_train_windows:].copy()
    return tr, ts


hfl_recon_rr = {}
train_actuals_by_h = {}

for H_IDX, target_col in enumerate(forecast_horizon):
    train_frames = []
    test_frames = []

    for nid in node_order:
        sub = nodes[nid].df.copy()
        tr_df, ts_df = split_node_df(sub, args)

        ts_out = ts_df[['unique_id', 'ds', target_col]].copy()
        ts_out.columns = ['unique_id', 'ds', 'y']
        ts_out['base'] = np.asarray(nodes[nid].base_forecast)[:, H_IDX]

        for method, arr in nodes[nid].reconciled_forecasts.items():
            ts_out[method] = np.asarray(arr)[:, H_IDX]

        test_frames.append(ts_out)

        tr_out = tr_df[['unique_id', 'ds', target_col]].copy()
        tr_out.columns = ['unique_id', 'ds', 'y']
        train_frames.append(tr_out)

    hfl_recon_rr[H_IDX] = pd.concat(test_frames, ignore_index=True)
    train_actuals_by_h[H_IDX] = pd.concat(train_frames, ignore_index=True)

hfl_recon_rr[0].head()


## Evaluation: level-wise MAE, MASE, and RMSSE

This section evaluates each method by hierarchy level. The MASE and RMSSE denominators are computed from each node's training-set true values (`node.y2`), not from the test set.


In [ ]:
import time


def _as_2d(arr):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr[:, None]
    return arr


def _target_cols(args):
    return ['y'] + [f'post_{i}' for i in range(1, int(args.fh) + 1)]


def split_node_df(df_one, args):
    time_col = getattr(args, 'time_col', 'ds')
    sub = df_one.sort_values(time_col).copy().reset_index(drop=True)

    n_windows = len(sub)
    n_raw = n_windows + int(args.lags) + int(args.fh)
    n_train_raw = int((1 - float(args.test_ratio)) * n_raw)
    n_train_windows = n_train_raw - int(args.lags) - int(args.fh)

    if n_train_windows <= 0 or n_train_windows >= n_windows:
        raise ValueError(
            f'Bad split for node: n_windows={n_windows}, n_train_windows={n_train_windows}'
        )

    tr = sub.iloc[:n_train_windows].copy()
    ts = sub.iloc[n_train_windows:].copy()
    return tr, ts


def make_naive_forecast(node, args):
    """Simple random-walk naive forecast: use the latest available lag, lags_1."""
    _, ts_df = split_node_df(node.df, args)
    y_true = _as_2d(node.test_true)
    if 'lags_1' not in ts_df.columns:
        raise KeyError("Cannot compute naive forecast because 'lags_1' is missing.")
    naive_1d = ts_df['lags_1'].to_numpy(dtype=np.float64)
    if len(naive_1d) != y_true.shape[0]:
        raise ValueError(
            f'Naive length mismatch: naive={len(naive_1d)}, y_true={y_true.shape[0]}'
        )
    return np.repeat(naive_1d[:, None], y_true.shape[1], axis=1)


def _safe_scale_denominators(train_true, eps=1e-12):
    train_true = _as_2d(train_true)
    if train_true.shape[0] <= 1:
        return np.nan, np.nan

    diff = np.diff(train_true, axis=0)
    mase_denom = np.nanmean(np.abs(diff))
    rmsse_denom = np.nanmean(diff ** 2)

    if (not np.isfinite(mase_denom)) or mase_denom <= eps:
        mase_denom = np.nan
    if (not np.isfinite(rmsse_denom)) or rmsse_denom <= eps:
        rmsse_denom = np.nan

    return float(mase_denom), float(rmsse_denom)


def compute_metrics_by_level(nodes, layout, args, include_naive=True, eps=1e-12):
    rows = []
    role_order = {'cloud': 1, 'edge': 2, 'client': 3}

    for nid in layout.node_ids:
        node = nodes[nid]
        role = layout.role_by_id[nid]
        unique_id = str(node.series_name)
        level = int(unique_id.count('/') + 1)

        y_true = _as_2d(node.test_true)
        train_true = _as_2d(node.y2)
        mase_denom, rmsse_denom = _safe_scale_denominators(train_true, eps=eps)

        method_forecasts = {'base': node.base_forecast}
        if include_naive:
            method_forecasts['naive'] = make_naive_forecast(node, args)
        method_forecasts.update(node.reconciled_forecasts)

        for method, forecast in method_forecasts.items():
            y_pred = _as_2d(forecast)
            if y_pred.shape != y_true.shape:
                raise ValueError(
                    f'Shape mismatch for node={nid}, method={method}: '
                    f'y_true={y_true.shape}, y_pred={y_pred.shape}'
                )

            error = y_true - y_pred
            mae = float(np.nanmean(np.abs(error)))
            mse = float(np.nanmean(error ** 2))
            mase = mae / mase_denom if np.isfinite(mase_denom) else np.nan
            rmsse = np.sqrt(mse / rmsse_denom) if np.isfinite(rmsse_denom) else np.nan

            rows.append({
                'node_id': int(nid),
                'unique_id': unique_id,
                'level': level,
                'role': role,
                'role_order': role_order.get(role, level),
                'method': str(method),
                'MAE': mae,
                'MASE': float(mase) if np.isfinite(mase) else np.nan,
                'RMSSE': float(rmsse) if np.isfinite(rmsse) else np.nan,
            })

    per_node_metrics = pd.DataFrame(rows)

    metrics_by_level = (
        per_node_metrics
        .groupby(['level', 'role', 'role_order', 'method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['role_order', 'level', 'method'])
        .drop(columns=['role_order'])
        .reset_index(drop=True)
    )

    overall_metrics = (
        per_node_metrics
        .groupby(['method'], as_index=False)
        .agg(
            MAE=('MAE', 'mean'),
            MASE=('MASE', 'mean'),
            RMSSE=('RMSSE', 'mean'),
            n_nodes=('node_id', 'nunique'),
        )
        .sort_values(['method'])
        .reset_index(drop=True)
    )

    return per_node_metrics, metrics_by_level, overall_metrics


def build_forecast_table_hfl(nodes, layout, args, include_naive=True):
    """Build a per-series forecast table with base, reconciled, and naive forecasts."""
    rows = []
    target_cols = _target_cols(args)
    time_col = getattr(args, 'time_col', 'ds')

    for nid in layout.node_ids:
        node = nodes[nid]
        _, ts_df = split_node_df(node.df, args)
        y_true = _as_2d(node.test_true)
        base = _as_2d(node.base_forecast)
        naive = make_naive_forecast(node, args) if include_naive else None
        rec = {method: _as_2d(arr) for method, arr in node.reconciled_forecasts.items()}

        for h_idx, target_col in enumerate(target_cols):
            for t_idx, (_, row) in enumerate(ts_df.iterrows()):
                out = {
                    'node_id': int(nid),
                    'unique_id': str(node.series_name),
                    'role': layout.role_by_id[nid],
                    'ds': row[time_col],
                    'horizon_index': int(h_idx),
                    'target_col': target_col,
                    'y_true': float(y_true[t_idx, h_idx]),
                    'base': float(base[t_idx, h_idx]),
                }
                if include_naive:
                    out['naive'] = float(naive[t_idx, h_idx])
                for method, arr in rec.items():
                    out[str(method)] = float(arr[t_idx, h_idx])
                rows.append(out)

    return pd.DataFrame(rows)


def save_hfl_outputs(output_prefix, nodes, layout, args, per_node_metrics,
                     metrics_by_level, overall_metrics, cloud_round_logs,
                     edge_round_logs, timings):
    """Save forecasts, per-series metrics, aggregate metrics, round logs, and time costs."""
    forecast_table = build_forecast_table_hfl(nodes, layout, args, include_naive=True)
    timing_df = make_timing_df(timings)

    paths = {
        'forecasts': f'{output_prefix}_forecasts.csv',
        'per_node_metrics': f'{output_prefix}_per_node_metrics.csv',
        'metrics_by_level': f'{output_prefix}_metrics_by_level.csv',
        'overall_metrics': f'{output_prefix}_overall_metrics.csv',
        'cloud_round_logs': f'{output_prefix}_cloud_round_logs.csv',
        'edge_round_logs': f'{output_prefix}_edge_round_logs.csv',
        'timing': f'{output_prefix}_timing.csv',
    }

    forecast_table.to_csv(paths['forecasts'], index=False)
    per_node_metrics.to_csv(paths['per_node_metrics'], index=False)
    metrics_by_level.to_csv(paths['metrics_by_level'], index=False)
    overall_metrics.to_csv(paths['overall_metrics'], index=False)
    pd.DataFrame(cloud_round_logs).to_csv(paths['cloud_round_logs'], index=False)
    pd.DataFrame(edge_round_logs).to_csv(paths['edge_round_logs'], index=False)
    timing_df.to_csv(paths['timing'], index=False)

    return paths, forecast_table, timing_df


_eval_t0 = time.perf_counter()
per_node_metrics, metrics_by_level, overall_metrics = compute_metrics_by_level(
    nodes, layout, args, include_naive=True
)
timings['evaluation_sec'] = time.perf_counter() - _eval_t0
timings['total_sec'] = (
    timings.get('setup_sec', 0.0)
    + timings.get('fl_training_sec', 0.0)
    + timings.get('reconciliation_sec', 0.0)
    + timings.get('evaluation_sec', 0.0)
)

output_prefix = str((PROJECT_ROOT / 'fhf_dedup_gef2017' / f'fh{args.fh + 1}_HFF').resolve())
from PPHF_modules.fhf.client_sampling import sampled_output_dir
_sampling_seed = args.seed if args.sampling_seed is None else args.sampling_seed
output_prefix = str(sampled_output_dir(Path(output_prefix).parent, args.sampling_rate, _sampling_seed) / Path(output_prefix).name)
Path(output_prefix).parent.mkdir(parents=True, exist_ok=True)
output_paths, forecast_table, timing_df = save_hfl_outputs(
    output_prefix=output_prefix,
    nodes=nodes,
    layout=layout,
    args=args,
    per_node_metrics=per_node_metrics,
    metrics_by_level=metrics_by_level,
    overall_metrics=overall_metrics,
    cloud_round_logs=cloud_round_logs,
    edge_round_logs=edge_round_logs,
    timings=timings,
)

result['timings'] = timings
result['per_node_metrics'] = per_node_metrics
result['metrics_by_level'] = metrics_by_level
result['overall_metrics'] = overall_metrics
result['forecast_table'] = forecast_table
result['output_paths'] = output_paths

print('Saved outputs:')
for name, path in output_paths.items():
    print(f'  {name}: {path}')

metrics_by_level


In [ ]:
# Optional: overall average across all hierarchy nodes
overall_metrics


In [ ]:
# Final timing table, saved by save_hfl_outputs(...).
timing_df


## Reusable event-level DP O2MR artifacts

This export stores the non-private reconciliation matrices and training-derived time-slot bounds needed for later event-level DP releases.


In [ ]:
# Save reusable event-level DP O2MR artifacts.
# The training run above remains non-private. These files allow event-level
# noise to be applied later without retraining the forecasting model.
from PPHF_modules.fhf.training_artifacts import (
    save_or_validate_shared_bounds,
    save_training_artifacts,
)


# Post-training export only: these reads do not update model weights, forecasts,
# reconciliation matrices, random state, or evaluation metrics.
training_artifacts_path = save_training_artifacts(
    nodes=nodes,
    cid2series=cid2series,
    node_order=list(layout.node_ids),
    output_path=f'{output_prefix}_training_artifacts.npz',
)
with np.load(training_artifacts_path, allow_pickle=False) as stored:
    shared_bounds_paths = save_or_validate_shared_bounds(
        artifacts={key: stored[key] for key in stored.files},
        artifact_dir=Path(training_artifacts_path).parent,
        forecast_horizon_count=int(args.fh) + 1,
        slots_per_day=int(args.dp_slots_per_day),
        lower_quantile=float(args.dp_lower_quantile),
        upper_quantile=float(args.dp_upper_quantile),
        iqr_margin=float(args.dp_iqr_margin),
        physical_lower=args.dp_physical_lower,
    )


def save_event_dp_o2mr_artifacts(
    output_prefix,
    framework,
    nodes,
    node_order,
    cid2series,
    role_by_id,
    Ps,
    args,
    completed_rounds,
    base_forecasts_path,
):
    node_order = [int(nid) for nid in node_order]
    sender_columns = [f'sender_col_{idx:04d}' for idx in range(len(node_order))]

    node_map_rows = []
    for matrix_index, nid in enumerate(node_order):
        node_map_rows.append({
            'matrix_index': int(matrix_index),
            'node_id': int(nid),
            'unique_id': str(cid2series[nid]),
            'role': str(role_by_id[nid]),
            'sender_column': sender_columns[matrix_index],
        })
    node_map_df = pd.DataFrame(node_map_rows)
    node_map_path = f'{output_prefix}_o2mr_node_map.csv'
    node_map_df.to_csv(node_map_path, index=False)

    p_paths = {}
    for method, raw_tensor in Ps.items():
        tensor = np.asarray(raw_tensor, dtype=np.float64)
        if tensor.ndim == 2:
            tensor = tensor[None, :, :]
        expected = (len(node_order), len(node_order))
        if tensor.ndim != 3 or tensor.shape[1:] != expected:
            raise ValueError(
                f'Unexpected P shape for {method}: {tensor.shape}; '
                f'expected (H, {expected[0]}, {expected[1]}).'
            )

        method_frames = []
        for horizon_index in range(tensor.shape[0]):
            frame = pd.DataFrame(tensor[horizon_index], columns=sender_columns)
            frame.insert(0, 'recipient_unique_id', [str(cid2series[nid]) for nid in node_order])
            frame.insert(0, 'recipient_node_id', node_order)
            frame.insert(0, 'recipient_matrix_index', np.arange(len(node_order), dtype=int))
            frame.insert(0, 'horizon_index', int(horizon_index))
            frame.insert(0, 'method', str(method))
            method_frames.append(frame)

        safe_method = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(method))
        p_path = f'{output_prefix}_o2mr_P_{safe_method}.csv'
        pd.concat(method_frames, ignore_index=True).to_csv(p_path, index=False)
        p_paths[f'P_{method}'] = p_path

    # All frameworks and training modes reference one method-independent
    # release calibrated from the common local training targets.
    bounds_path = str(shared_bounds_paths['bounds'])
    slot_map_path = str(shared_bounds_paths['time_slot_map'])

    config_path = f'{output_prefix}_event_dp_o2mr_config.csv'
    pd.DataFrame([{
        'framework': str(framework),
        'forecast_horizon_count': int(args.fh) + 1,
        'num_rounds': int(args.num_rounds),
        'min_rounds': int(args.min_rounds),
        'completed_rounds': int(completed_rounds),
        'dp_enabled_during_training_run': bool(args.dp_enabled),
        'dp_privacy_level': str(args.dp_privacy_level),
        'dp_mechanism': str(args.dp_mechanism),
        'dp_epsilon': float(args.dp_epsilon),
        'dp_delta': float(args.dp_delta),
        'dp_bounds_source': 'common_local_training_targets',
        'dp_bounds_scope': str(args.dp_bounds_scope),
        'dp_slots_per_day': int(args.dp_slots_per_day),
        'dp_lower_quantile': float(args.dp_lower_quantile),
        'dp_upper_quantile': float(args.dp_upper_quantile),
        'dp_iqr_margin': float(args.dp_iqr_margin),
        'dp_physical_lower': args.dp_physical_lower,
        'dp_aware_reconciliation': bool(args.dp_aware_reconciliation),
        'P_uses_nonprivate_residual_covariance': True,
        'bounds_require_public_or_permitted_calibration': True,
        'base_forecasts_csv': str(base_forecasts_path),
        'node_map_csv': node_map_path,
        'bounds_csv': Path(bounds_path).name,
        'time_slot_map_csv': Path(slot_map_path).name,
    }]).to_csv(config_path, index=False)

    paths = {
        'o2mr_node_map': node_map_path,
        'event_dp_train_bounds': bounds_path,
        'event_dp_time_slot_map': slot_map_path,
        'event_dp_o2mr_config': config_path,
    }
    paths.update(p_paths)
    return paths


event_dp_o2mr_paths = save_event_dp_o2mr_artifacts(
    output_prefix=output_prefix,
    framework='HFF',
    nodes=nodes,
    node_order=list(layout.node_ids),
    cid2series=cid2series,
    role_by_id=layout.role_by_id,
    Ps=result['Ps'],
    args=args,
    completed_rounds=len(cloud_round_logs),
    base_forecasts_path=output_paths['forecasts'],
)
output_paths.update(event_dp_o2mr_paths)
output_paths['training_artifacts'] = str(training_artifacts_path)
output_paths['shared_event_dp_bounds_metadata'] = str(
    shared_bounds_paths['metadata']
)
result['output_paths'] = output_paths

print('Saved event-level DP O2MR artifacts:')
display(pd.DataFrame([
    {'artifact': key, 'path': value}
    for key, value in event_dp_o2mr_paths.items()
]))
